1. Выбор датасета:

Выбор пал на предложенный авторами задания датасет - https://www.kaggle.com/code/aryantiwari123/iris-species/input

Колонки выбранного датасета:
- Id (айди)
- SepalLengthCm (длина "чашелистика")
- SepalWidthCm (ширина "чашелистица")
- PetalLengthCm (длина лепестка)
- PetalWidthCm (ширина лепестка)
- Species (вид)

Выбрал его, потому что:
- он был первый в списке предложенных для обработки
- ну, собственно, и всё

2. Первичный анализ данных датасета Iris Species:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCv
from sklearn.neighbors import KNeighborsClassifier

sns.set_style('dark')

df = pd.read_csv('Iris.csv')

In [90]:
df.head(3)

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa


In [70]:
df.tail(3)

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
147,148,6.5,3.0,5.2,2.0,Iris-virginica
148,149,6.2,3.4,5.4,2.3,Iris-virginica
149,150,5.9,3.0,5.1,1.8,Iris-virginica


In [71]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             150 non-null    int64  
 1   SepalLengthCm  150 non-null    float64
 2   SepalWidthCm   150 non-null    float64
 3   PetalLengthCm  150 non-null    float64
 4   PetalWidthCm   150 non-null    float64
 5   Species        150 non-null    str    
dtypes: float64(4), int64(1), str(1)
memory usage: 7.2 KB


In [96]:
df.describe()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm
count,150.000000,150.000000,150.000000,150.000000,150.000000
mean,75.500000,5.843333,3.054000,3.758667,1.198667
std,43.445368,0.828066,0.433594,1.764420,0.763161
min,1.000000,4.300000,2.000000,1.000000,0.100000
25%,38.250000,5.100000,2.800000,1.600000,0.300000
50%,75.500000,5.800000,3.000000,4.350000,1.300000
75%,112.750000,6.400000,3.300000,5.100000,1.800000
max,150.000000,7.900000,4.400000,6.900000,2.500000


In [95]:
df.select_dtypes(include=['int64', 'float64']).mean()

Id               75.500000
SepalLengthCm     5.843333
SepalWidthCm      3.054000
PetalLengthCm     3.758667
PetalWidthCm      1.198667
dtype: float64

In [94]:
df.select_dtypes(include=['int64', 'float64']).median()

Id               75.50
SepalLengthCm     5.80
SepalWidthCm      3.00
PetalLengthCm     4.35
PetalWidthCm      1.30
dtype: float64

In [89]:
df.isnull().sum()

Id               0
SepalLengthCm    0
SepalWidthCm     0
PetalLengthCm    0
PetalWidthCm     0
Species          0
dtype: int64

In [92]:
df.duplicated().sum()

np.int64(0)

In [93]:
df.shape

(150, 6)

In [97]:
df.select_dtypes(include=['int64', 'float64']).var()

Id               1887.500000
SepalLengthCm       0.685694
SepalWidthCm        0.188004
PetalLengthCm       3.113179
PetalWidthCm        0.582414
dtype: float64

In [98]:
df.select_dtypes(include=['int64', 'float64']).skew()

Id               0.000000
SepalLengthCm    0.314911
SepalWidthCm     0.334053
PetalLengthCm   -0.274464
PetalWidthCm    -0.104997
dtype: float64

In [99]:
df.select_dtypes(include=['int64', 'float64']).kurtosis()

Id              -1.200000
SepalLengthCm   -0.552064
SepalWidthCm     0.290781
PetalLengthCm   -1.401921
PetalWidthCm    -1.339754
dtype: float64

In [36]:
df['Species'].value_counts(normalize=True)

Species
Iris-setosa        0.333333
Iris-versicolor    0.333333
Iris-virginica     0.333333
Name: proportion, dtype: float64

In [91]:
df_melted = df.melt(var_name='Attribute', value_name='Value')
fig = px.box(
    df_melted,
    x='Attribute',
    y='Value',
    template='plotly_dark'
)

fig.show()

In [92]:
df_cols = df.select_dtypes(include=['float64', 'int64']).columns.to_list()
for i in df_cols:
    fig = px.histogram(
        df,
        x=i,
        height=500,
        width=800,
        nbins=50,
    )
    fig.update_layout(
        yaxis_title='Frequency'
    )
    fig.update_traces(
        marker_line_color='black', 
        marker_line_width=1.5
    )
    fig.show()

Исходя из проведённого анализа, можно сделать следующие выводы:
- датасет маленький (150 строк на 6 колонок);
- выбросов практически нет;
- нет "отсутствующих" значений, пропуски заполнять не нужно;
- единственная текстовая колонка - название вида (Species), исходя из этого именно она будет являться target переменной;
- распределение классов target переменной идеально равномерно (0.(3) для каждого класса);
- исходя из предложенных выше графиков, ряд признаков сильно соответствует нормальному распределению (исключение - SepalWidthCm).Наблюдается как мультимодальность (2 и более пика), так и реже встречающееся среднее значение (отрицательный эксцесс у ряда признаков);

Вывод: единственная проблема датсета - его размер. В случае с KNN 150 объектов мало для уверенности в получении верно предсказывающей результат модели.

3. Подготовка данных:

Масштабирование признаков. В случае с KNN зачастую используют стандартное отклонение (в sklearn это StandartScaler). 

Нормализовать данные нужно, чтобы одни признаки не влияли на выбор ответа обучаемой моделью сильнее, чем другие (если один признак меняется в диапазоне [0,1], а другой [0,100], на расстояние второй будет влиять сильнее при достаточном изменение показателя (изменение на 3 пункта, что есть 3% от всего диапазона значений второго признака влияют КАК МИНИМУМ в 3 раза сильнее на итоговый ответ модели, чем первый признак на ВСЁМ диапазоне им принимаемых значений)). Нормализация данных позволяет минимизировать "влияние" подобных отклонений на итоговый ответ, оставив лишь реальное "влияние" признаков.

Ох я и намудрил...)))

Теперь определим target:
- интуитивно гораздо правильнее угадывать не длину или ширину листа (хотя это было бы интересно...), а название вида;
- единственный категориальный признак, не представленный числом в данный момент, является название вида;

Вывод: target переменная - Species

Разделение на train и test. 

Так как у нас 3 класса, каждый занимает в исходном датасете ровно треть, логично поделить датасет следующим образом: 70% выборка на тренировку - 30% тест (в противном случае, если на тест оставить меньше значений, результат будет сильно коррелироваться в зависимости от исходных данных (если повезёт, может и всё угадаем, а может и, наоборот, точность будет слишком низкой)). Далее - нужно сделать равное разбиение по классам (чтобы в train попали все 3 класса, не получится просто взять первые 70% строк в train, сделаем хитрее).

Путём недолгих подсчётов:
В каждом классе 50 объектов, каждый класс представлен в test выборке 50/100*30 = 15-ю объектами. Так формируем test, остальное - train выборку.

На тестовой выборке подбор параметров осуществлять не будем, так как в таком случае есть вероятность, что модель просто "запомнит", какому набору входных данных какой вид соответствует, но при этом сама не научится решать задачу.

Кодирование данных.

Наконец, кодирование данных. Так как в нашем датасете всего 1 категориальный признак нуждается в кодировке, но именно он и является target переменной, мы не будем кодировать данные при поставленной задаче и данном датасете.

In [93]:
Y = df['Species']
X = df.drop(['Species', 'Id'], axis=1)

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.3,
    random_state=42,
    stratify=Y
)


In [94]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

После разбиения и стандартизации данных посмотрим, что получили:

In [95]:
X_train_scaled.mean(axis=0)

array([ 3.00288894e-16, -1.16983143e-15, -4.56777473e-16,  1.01506105e-16])

In [96]:
X_test_scaled.mean(axis=0)

array([-0.11643861,  0.02599252, -0.05081417, -0.02621151])

Очень близко к нулю на train, так как на нём всё считали, на тесте погрешность немного больше, но тоже около нуля.

In [97]:
X_train_scaled.std(axis=0)

array([1., 1., 1., 1.])

In [98]:
X_test_scaled.std(axis=0)

array([0.85754489, 0.8452672 , 0.9691546 , 0.93583444])

На train std=1, на test к нему стремится - следовательно всё замечательно.

4. Обучение KNN;
5. Подбор гиперпараметров;

Наконец, перейдём к самому интересному: обучение KNN и подбору наилучшего набора гиперпараметров. Так как наша модель достаточно мала, а знаний в интуитивном (и, конечно же, не только интуитивном) подборе гиперпараметров у меня недостаточно, чтобы сделать это легко и просто, мы переберём все возможные варианты (в дальнейшем, конечно, такую практику будем избегать, но сейчас, в рамках обучения и в качестве улучшения насмотренности, мы это сделаем) гиперпараметры.

Для этого воспользуемся GridSearchCv:

In [ ]:
values = range(1, 20)
metrics = ['euclidean', 'manhattan', 'chebyshev']
weights_types = ['uniform', 'distance']

for weight_type in weights_types:
    for metric in metrics:
        scores = []
        for value in  values:
            knn = KNeighborsClassifier(n_neighbors=value, weights=weight_type, metric=metric)
            knn.fit(X_train_scaled, Y_train)
            scores.append(knn.score(X_test_scaled, Y_test))
        df_scores = pd.DataFrame({
            'k': list(values),
            'accuracy': scores
        }) 
        fig = px.line(
            df_scores,
            x='k',
            y='accuracy',
            title=f'KNN: metric={metric}, weights={weight_type}'
        )
        fig.show()

SyntaxError: invalid syntax (3746575256.py, line 13)